# 05 - The go / no-go simulator

`src/simulator.py`: among the comparable games, the **probability of profitability** is the share of those whose net revenue exceeds the studio's budget.
Thresholds: GO from 50 %, RISKY from 25 to 50 %, NO-GO below.

In [1]:
import sys
sys.path.append('..')

import pandas as pd
from src.cleaning import load_clean
from src.estimation import add_estimates
from src.simulator import GameConcept, recommendation

df = add_estimates(load_clean('../data/games_clean.csv'))

## 1. Test the basic functions

In [2]:
for p in [0.8, 0.5, 0.3, 0.25, 0.1]:
    print(p, '->', recommendation(p))

0.8 -> GO
0.5 -> GO
0.3 -> RISKY
0.25 -> RISKY
0.1 -> NO-GO


## 2. Sanity check: Indie + Roguelike, $15, $50,000 then $500,000

In [3]:
result = GameConcept(['Indie', 'Roguelike'], 15, 50000).evaluate(df)
result

{'tags': ['Indie', 'Roguelike'],
 'price': 15.0,
 'budget': 50000.0,
 'n_comparables': 625,
 'revenue_range': (2704.59, 16782.444, 100447.45199999999),
 'probabilities': {'low': 0.2544, 'mid': 0.344, 'high': 0.4016},
 'recommendation': 'RISKY',
 'recommendations': {'low': 'RISKY', 'mid': 'RISKY', 'high': 'RISKY'},
 'recommended_price': '20-30$',
 'warning': None}

In [4]:
for budget in [50000, 500000]:
    concept = GameConcept(['Indie', 'Roguelike'], 15, budget)
    concept.evaluate(df)
    print(concept.summary())

625 comparable games, 34% covered $50,000: RISKY. Recommended price: 20-30$.


625 comparable games, 10% covered $500,000: NO-GO. Recommended price: 20-30$.


The budget goes from $50,000 to $500,000: the probability drops from 34 % to 10 %, and the verdict goes from RISKY to NO-GO. As expected.

## 3. Our running example: 2 people, $60,000, roguelike or cozy game at $15?

In [5]:
rows = []
running_example = {'A - Roguelike': ['Indie', 'Roguelike'], 'B - Cozy': ['Indie', 'Cozy']}
for name, tags in running_example.items():
    concept = GameConcept(tags, 15, 60000)
    r = concept.evaluate(df)
    print(name, ':', concept.summary())
    for scenario in ['low', 'mid', 'high']:
        rows.append({'concept': name, 'scenario': scenario,
                     'probability': round(r['probabilities'][scenario], 2),
                     'verdict': r['recommendations'][scenario]})
pd.DataFrame(rows)

A - Roguelike : 625 comparable games, 32% covered $60,000: RISKY. Recommended price: 20-30$.


B - Cozy : 119 comparable games, 40% covered $60,000: RISKY. Recommended price: 15-20$.


,concept,scenario,probability,verdict
0,A - Roguelike,low,0.24,NO-GO
1,A - Roguelike,mid,0.32,RISKY
2,A - Roguelike,high,0.37,RISKY
3,B - Cozy,low,0.29,RISKY
4,B - Cozy,mid,0.40,RISKY
5,B - Cozy,high,0.45,RISKY


**Answer to the running example**: both concepts are **RISKY** at $60,000, but the **cozy game** has better odds (40 % against 32 %) and a higher revenue range.
**Sensitivity**: the cozy game stays RISKY in all 3 scenarios, while the roguelike **drops to NO-GO** in the low scenario (24 %, just below the 25 % threshold). The cozy concept is therefore the most robust to estimation errors.
Caution: the cozy niche has few comparable games (about 120) and its releases have tripled since 2021-22.

## 4. Three contrasting concepts

In [6]:
tests = {
    'Small budget, growing niche (Open World Survival Craft, $15, $20,000)': (['Open World Survival Craft'], 15, 20000),
    'Big budget, saturated niche (Clicker, $5, $300,000)': (['Clicker'], 5, 300000),
    'Almost empty niche (Roguelike Deckbuilder + Trains, $15, $60,000)': (['Roguelike Deckbuilder', 'Trains'], 15, 60000),
}
for name, args in tests.items():
    tags, price, budget = args
    concept = GameConcept(tags, price, budget)
    concept.evaluate(df)
    print(name, '\n   ', concept.summary())

Small budget, growing niche (Open World Survival Craft, $15, $20,000) 
    216 comparable games, 54% covered $20,000: GO. Recommended price: 30$+.


Big budget, saturated niche (Clicker, $5, $300,000) 
    724 comparable games, 1% covered $300,000: NO-GO. Recommended price: 30$+.


Almost empty niche (Roguelike Deckbuilder + Trains, $15, $60,000) 
    1 comparable games, 100% covered $60,000: RISKY. Recommended price: not determined. Warning: Only 1 comparable games (recommended minimum: 30): the result is not reliable.


The results make sense: GO for the growing niche with a small budget, NO-GO for the clicker with a big budget, and a warning (verdict capped at RISKY) when there is only one comparable game.

## 5. Edge cases: no crash

In [7]:
for tags, price, budget in [([], 15, 60000), (['Indie'], -5, 60000), (['Indie'], 15, 0)]:
    try:
        GameConcept(tags, price, budget)
    except ValueError as error:
        print('Input cleanly refused:', error)

empty = GameConcept(['Tag that does not exist'], 15, 60000)
empty.evaluate(df)
print(empty.summary())

Input cleanly refused: Choose at least one tag.
Input cleanly refused: The price must be strictly positive.
Input cleanly refused: The budget must be strictly positive.


0 comparable games, 0% covered $60,000: NO-GO. Recommended price: not determined. Warning: No comparable game: this concept cannot be evaluated.
